In [0]:
# ==========================================================
# Célula 1 — Configuração
# ==========================================================

from pyspark.sql import functions as F

CATALOGO = "dbw_fiap_techchallenge_prod_7405607602465611"
SCHEMA   = "default"

TABELA_GOLD_UF  = f"{CATALOGO}.{SCHEMA}.gold_indicador_uf"
TABELA_DEST_NAC = f"{CATALOGO}.{SCHEMA}.gold_indicador_nacional"

print(f"Gold UF      : {TABELA_GOLD_UF}")
print(f"Destino NAC  : {TABELA_DEST_NAC}")

In [0]:
# ==========================================================
# Célula 2 — Agregação nacional derivada da Gold UF
#
# Autossuficiente: define a configuração no topo.
# CORREÇÃO 1: o horizonte temporal é o da Gold UF (2023-2025).
#             Anos sem medição entram como sem_dado, não desaparecem.
# CORREÇÃO 2: taxa E meta usam o MESMO peso (percentual_participacao).
#             Comparar taxa ponderada com meta simples distorcia o gap.
# ==========================================================

from pyspark.sql import functions as F

CATALOGO = "dbw_fiap_techchallenge_prod_7405607602465611"
SCHEMA   = "default"
TABELA_GOLD_UF = f"{CATALOGO}.{SCHEMA}.gold_indicador_uf"

df_uf = spark.table(TABELA_GOLD_UF)

# Peso único para taxa e meta. Fallback 1.0 quando ausente.
df_uf_p = df_uf.withColumn(
    "peso", F.coalesce(F.col("percentual_participacao"), F.lit(1.0))
)

df_nac = (
    df_uf_p
    .groupBy("ano")
    .agg(
        # Taxa nacional: média ponderada apenas das UFs com medição
        F.round(
            F.sum(F.when(F.col("taxa_observada").isNotNull(),
                         F.col("taxa_observada") * F.col("peso")))
            / F.sum(F.when(F.col("taxa_observada").isNotNull(), F.col("peso"))),
            4
        ).alias("taxa_observada"),

        # Meta nacional: média ponderada apenas das UFs com meta
        F.round(
            F.sum(F.when(F.col("meta_do_ano").isNotNull(),
                         F.col("meta_do_ano") * F.col("peso")))
            / F.sum(F.when(F.col("meta_do_ano").isNotNull(), F.col("peso"))),
            4
        ).alias("meta_do_ano"),

        F.round(F.avg("media_portugues"), 4).alias("media_portugues"),
        F.count("*").alias("qtd_ufs_total"),
        F.sum(F.when(F.col("taxa_observada").isNotNull(), 1).otherwise(0)).alias("qtd_ufs_com_dado"),
        F.sum(F.when(F.col("meta_do_ano").isNotNull(), 1).otherwise(0)).alias("qtd_ufs_com_meta"),
        F.sum(F.when(F.col("status_meta") == "atingiu", 1).otherwise(0)).alias("qtd_ufs_atingiu"),
        F.sum(F.when(F.col("status_meta") == "abaixo", 1).otherwise(0)).alias("qtd_ufs_abaixo"),
        F.sum(F.when(F.col("status_meta") == "sem_meta", 1).otherwise(0)).alias("qtd_ufs_sem_meta"),
        F.sum(F.when(F.col("status_meta") == "sem_dado", 1).otherwise(0)).alias("qtd_ufs_sem_dado"),
    )
)

print(f"Anos na Gold nacional: {df_nac.count()}")
df_nac.orderBy("ano").show(truncate=False)

In [0]:
# ==========================================================
# Célula 3 — Enriquecimento e status nacional
#
# Autossuficiente: reconstrói df_nac no topo.
# ==========================================================

from pyspark.sql import functions as F

CATALOGO = "dbw_fiap_techchallenge_prod_7405607602465611"
SCHEMA   = "default"
TABELA_GOLD_UF = f"{CATALOGO}.{SCHEMA}.gold_indicador_uf"

df_uf = spark.table(TABELA_GOLD_UF)
df_uf_p = df_uf.withColumn(
    "peso", F.coalesce(F.col("percentual_participacao"), F.lit(1.0))
)

df_nac = (
    df_uf_p
    .groupBy("ano")
    .agg(
        F.round(
            F.sum(F.when(F.col("taxa_observada").isNotNull(),
                         F.col("taxa_observada") * F.col("peso")))
            / F.sum(F.when(F.col("taxa_observada").isNotNull(), F.col("peso"))),
            4
        ).alias("taxa_observada"),
        F.round(
            F.sum(F.when(F.col("meta_do_ano").isNotNull(),
                         F.col("meta_do_ano") * F.col("peso")))
            / F.sum(F.when(F.col("meta_do_ano").isNotNull(), F.col("peso"))),
            4
        ).alias("meta_do_ano"),
        F.round(F.avg("media_portugues"), 4).alias("media_portugues"),
        F.count("*").alias("qtd_ufs_total"),
        F.sum(F.when(F.col("taxa_observada").isNotNull(), 1).otherwise(0)).alias("qtd_ufs_com_dado"),
        F.sum(F.when(F.col("meta_do_ano").isNotNull(), 1).otherwise(0)).alias("qtd_ufs_com_meta"),
        F.sum(F.when(F.col("status_meta") == "atingiu", 1).otherwise(0)).alias("qtd_ufs_atingiu"),
        F.sum(F.when(F.col("status_meta") == "abaixo", 1).otherwise(0)).alias("qtd_ufs_abaixo"),
        F.sum(F.when(F.col("status_meta") == "sem_meta", 1).otherwise(0)).alias("qtd_ufs_sem_meta"),
        F.sum(F.when(F.col("status_meta") == "sem_dado", 1).otherwise(0)).alias("qtd_ufs_sem_dado"),
    )
    .withColumn(
        "gap_pp",
        F.when(
            F.col("taxa_observada").isNotNull() & F.col("meta_do_ano").isNotNull(),
            F.round(F.col("taxa_observada") - F.col("meta_do_ano"), 2)
        ).otherwise(F.lit(None).cast("double"))
    )
    .withColumn(
        "status_meta",
        F.when(F.col("taxa_observada").isNull(), F.lit("sem_dado"))
         .when(F.col("meta_do_ano").isNull(), F.lit("sem_meta"))
         .when(F.col("taxa_observada") >= F.col("meta_do_ano"), F.lit("atingiu"))
         .otherwise(F.lit("abaixo"))
    )
    .withColumn(
        "pct_ufs_atingiu",
        F.when(
            F.col("qtd_ufs_com_dado") > 0,
            F.round(F.col("qtd_ufs_atingiu") / F.col("qtd_ufs_com_dado") * 100, 2)
        ).otherwise(F.lit(None).cast("double"))
    )
    .withColumn(
        "tem_dado_completo",
        (F.col("taxa_observada").isNotNull() & F.col("meta_do_ano").isNotNull()).cast("boolean")
    )
)

print(f"Registros na Gold nacional: {df_nac.count()}")
print("\nStatus por ano:")
df_nac.orderBy("ano").select(
    "ano", "taxa_observada", "meta_do_ano", "gap_pp", "status_meta",
    "qtd_ufs_total", "qtd_ufs_com_dado", "qtd_ufs_com_meta",
    "qtd_ufs_atingiu", "qtd_ufs_abaixo", "pct_ufs_atingiu"
).show(truncate=False)

In [0]:
# ==========================================================
# Célula 4 — Data Quality da Gold nacional
#
# Autossuficiente: reconstrói df_nac no topo.
# ==========================================================

from pyspark.sql import functions as F

CATALOGO = "dbw_fiap_techchallenge_prod_7405607602465611"
SCHEMA   = "default"
TABELA_GOLD_UF = f"{CATALOGO}.{SCHEMA}.gold_indicador_uf"

df_uf = spark.table(TABELA_GOLD_UF)
df_uf_p = df_uf.withColumn(
    "peso", F.coalesce(F.col("percentual_participacao"), F.lit(1.0))
)

df_nac = (
    df_uf_p
    .groupBy("ano")
    .agg(
        F.round(
            F.sum(F.when(F.col("taxa_observada").isNotNull(),
                         F.col("taxa_observada") * F.col("peso")))
            / F.sum(F.when(F.col("taxa_observada").isNotNull(), F.col("peso"))),
            4
        ).alias("taxa_observada"),
        F.round(
            F.sum(F.when(F.col("meta_do_ano").isNotNull(),
                         F.col("meta_do_ano") * F.col("peso")))
            / F.sum(F.when(F.col("meta_do_ano").isNotNull(), F.col("peso"))),
            4
        ).alias("meta_do_ano"),
        F.round(F.avg("media_portugues"), 4).alias("media_portugues"),
        F.count("*").alias("qtd_ufs_total"),
        F.sum(F.when(F.col("taxa_observada").isNotNull(), 1).otherwise(0)).alias("qtd_ufs_com_dado"),
        F.sum(F.when(F.col("meta_do_ano").isNotNull(), 1).otherwise(0)).alias("qtd_ufs_com_meta"),
        F.sum(F.when(F.col("status_meta") == "atingiu", 1).otherwise(0)).alias("qtd_ufs_atingiu"),
        F.sum(F.when(F.col("status_meta") == "abaixo", 1).otherwise(0)).alias("qtd_ufs_abaixo"),
        F.sum(F.when(F.col("status_meta") == "sem_meta", 1).otherwise(0)).alias("qtd_ufs_sem_meta"),
        F.sum(F.when(F.col("status_meta") == "sem_dado", 1).otherwise(0)).alias("qtd_ufs_sem_dado"),
    )
    .withColumn(
        "gap_pp",
        F.when(
            F.col("taxa_observada").isNotNull() & F.col("meta_do_ano").isNotNull(),
            F.round(F.col("taxa_observada") - F.col("meta_do_ano"), 2)
        ).otherwise(F.lit(None).cast("double"))
    )
    .withColumn(
        "status_meta",
        F.when(F.col("taxa_observada").isNull(), F.lit("sem_dado"))
         .when(F.col("meta_do_ano").isNull(), F.lit("sem_meta"))
         .when(F.col("taxa_observada") >= F.col("meta_do_ano"), F.lit("atingiu"))
         .otherwise(F.lit("abaixo"))
    )
    .withColumn(
        "pct_ufs_atingiu",
        F.when(
            F.col("qtd_ufs_com_dado") > 0,
            F.round(F.col("qtd_ufs_atingiu") / F.col("qtd_ufs_com_dado") * 100, 2)
        ).otherwise(F.lit(None).cast("double"))
    )
    .withColumn(
        "tem_dado_completo",
        (F.col("taxa_observada").isNotNull() & F.col("meta_do_ano").isNotNull()).cast("boolean")
    )
)

total_nac = df_nac.count()

nulos_chave = df_nac.filter(F.col("ano").isNull()).count()
duplicatas  = total_nac - df_nac.dropDuplicates(["ano"]).count()
status_inv  = df_nac.filter(
    ~F.col("status_meta").isin("atingiu", "abaixo", "sem_meta", "sem_dado")
).count()
gap_incons  = df_nac.filter(
    F.col("gap_pp").isNotNull() &
    (F.col("taxa_observada").isNull() | F.col("meta_do_ano").isNull())
).count()
taxa_fora   = df_nac.filter(
    F.col("taxa_observada").isNotNull() &
    ((F.col("taxa_observada") < 0) | (F.col("taxa_observada") > 100))
).count()
meta_fora   = df_nac.filter(
    F.col("meta_do_ano").isNotNull() &
    ((F.col("meta_do_ano") < 0) | (F.col("meta_do_ano") > 100))
).count()

# Consistência entre camadas: mesmo horizonte temporal da Gold UF
anos_uf  = {r[0] for r in spark.table(TABELA_GOLD_UF).select("ano").distinct().collect()}
anos_nac = {r[0] for r in df_nac.select("ano").distinct().collect()}
anos_faltando = anos_uf - anos_nac

print("=" * 55)
print("DATA QUALITY — gold_indicador_nacional")
print("=" * 55)
print(f"Registros totais       : {total_nac}")
print(f"Chave nula             : {nulos_chave}")
print(f"Duplicatas na chave    : {duplicatas}")
print(f"Status fora do domínio : {status_inv}")
print(f"Gap inconsistente      : {gap_incons}")
print(f"Taxa fora de 0-100     : {taxa_fora}")
print(f"Meta fora de 0-100     : {meta_fora}")
print(f"Anos                   : {sorted(anos_nac)}")
print(f"Anos na Gold UF        : {sorted(anos_uf)}")
print(f"Anos faltando          : {sorted(anos_faltando) if anos_faltando else 'nenhum'}")
print("=" * 55)

assert nulos_chave == 0, "FALHA: chave nula"
assert duplicatas == 0, "FALHA: duplicatas na chave"
assert status_inv == 0, "FALHA: status fora do domínio"
assert gap_incons == 0, "FALHA: gap inconsistente"
assert taxa_fora == 0, "FALHA: taxa fora de 0-100"
assert meta_fora == 0, "FALHA: meta fora de 0-100"
assert not anos_faltando, f"FALHA: anos ausentes em relação à Gold UF: {sorted(anos_faltando)}"
print("Todas as regras críticas passaram.")

In [0]:
# ==========================================================
# Célula 5 — Gravação e validação da Gold nacional
#
# Autossuficiente: reconstrói df_nac no topo.
# OPTIMIZE protegido por try/except.
# ==========================================================

from pyspark.sql import functions as F

CATALOGO = "dbw_fiap_techchallenge_prod_7405607602465611"
SCHEMA   = "default"
TABELA_GOLD_UF  = f"{CATALOGO}.{SCHEMA}.gold_indicador_uf"
TABELA_DEST_NAC = f"{CATALOGO}.{SCHEMA}.gold_indicador_nacional"

df_uf = spark.table(TABELA_GOLD_UF)
df_uf_p = df_uf.withColumn(
    "peso", F.coalesce(F.col("percentual_participacao"), F.lit(1.0))
)

df_nac = (
    df_uf_p
    .groupBy("ano")
    .agg(
        F.round(
            F.sum(F.when(F.col("taxa_observada").isNotNull(),
                         F.col("taxa_observada") * F.col("peso")))
            / F.sum(F.when(F.col("taxa_observada").isNotNull(), F.col("peso"))),
            4
        ).alias("taxa_observada"),
        F.round(
            F.sum(F.when(F.col("meta_do_ano").isNotNull(),
                         F.col("meta_do_ano") * F.col("peso")))
            / F.sum(F.when(F.col("meta_do_ano").isNotNull(), F.col("peso"))),
            4
        ).alias("meta_do_ano"),
        F.round(F.avg("media_portugues"), 4).alias("media_portugues"),
        F.count("*").alias("qtd_ufs_total"),
        F.sum(F.when(F.col("taxa_observada").isNotNull(), 1).otherwise(0)).alias("qtd_ufs_com_dado"),
        F.sum(F.when(F.col("meta_do_ano").isNotNull(), 1).otherwise(0)).alias("qtd_ufs_com_meta"),
        F.sum(F.when(F.col("status_meta") == "atingiu", 1).otherwise(0)).alias("qtd_ufs_atingiu"),
        F.sum(F.when(F.col("status_meta") == "abaixo", 1).otherwise(0)).alias("qtd_ufs_abaixo"),
        F.sum(F.when(F.col("status_meta") == "sem_meta", 1).otherwise(0)).alias("qtd_ufs_sem_meta"),
        F.sum(F.when(F.col("status_meta") == "sem_dado", 1).otherwise(0)).alias("qtd_ufs_sem_dado"),
    )
    .withColumn(
        "gap_pp",
        F.when(
            F.col("taxa_observada").isNotNull() & F.col("meta_do_ano").isNotNull(),
            F.round(F.col("taxa_observada") - F.col("meta_do_ano"), 2)
        ).otherwise(F.lit(None).cast("double"))
    )
    .withColumn(
        "status_meta",
        F.when(F.col("taxa_observada").isNull(), F.lit("sem_dado"))
         .when(F.col("meta_do_ano").isNull(), F.lit("sem_meta"))
         .when(F.col("taxa_observada") >= F.col("meta_do_ano"), F.lit("atingiu"))
         .otherwise(F.lit("abaixo"))
    )
    .withColumn(
        "pct_ufs_atingiu",
        F.when(
            F.col("qtd_ufs_com_dado") > 0,
            F.round(F.col("qtd_ufs_atingiu") / F.col("qtd_ufs_com_dado") * 100, 2)
        ).otherwise(F.lit(None).cast("double"))
    )
    .withColumn(
        "tem_dado_completo",
        (F.col("taxa_observada").isNotNull() & F.col("meta_do_ano").isNotNull()).cast("boolean")
    )
)

(df_nac.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_DEST_NAC))

print(f"Gravado em {TABELA_DEST_NAC}")

df_check = spark.table(TABELA_DEST_NAC)
print(f"Registros: {df_check.count()}")

print("\nVisão nacional por ano:")
df_check.orderBy("ano").select(
    "ano", "taxa_observada", "meta_do_ano", "gap_pp", "status_meta",
    "qtd_ufs_total", "qtd_ufs_com_dado", "qtd_ufs_com_meta",
    "qtd_ufs_atingiu", "qtd_ufs_abaixo", "pct_ufs_atingiu"
).show(truncate=False)

try:
    spark.sql(f"OPTIMIZE {TABELA_DEST_NAC} ZORDER BY (ano)")
    print("\nZORDER aplicado em ano.")
except Exception as e:
    print(f"\nZORDER não aplicado (não bloqueia a gravação): {e}")